# 1. Explicit example
## 1.1 Preparing the data

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("SparkByExamples")\
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.instances", "10") \
    .getOrCreate()
    # .config("spark.some.config.option", "config-value") 

In [ ]:
executor_memory = spark.sparkContext.getConf().get("spark.executor.memory")
print(f"Executor Memory: {executor_memory}")

In [ ]:
driver_memory = spark.sparkContext.getConf().get("spark.driver.memory")
print(f"driver Memory: {driver_memory}")

In [ ]:
spark.sparkContext.getConf().get("spark.executor.instances")

In [ ]:
executor_cores = spark.sparkContext.getConf().get("spark.executor.cores", "Not Set")
print(f"Executor Cores: {executor_cores}")

In [ ]:
for conf in spark.sparkContext.getConf().getAll():
    print(conf)

In [ ]:
BUCKET = "fgao-ensae"
FILE_KEY_S3 = "Data_spark/Recommendation/ratings.parquet/"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"

In [ ]:
s3_path

In [ ]:
# df = spark.read.csv(s3_path, header=True,)

In [ ]:
ratings = spark.read.parquet(s3_path)

In [ ]:
ratings.count()

In [ ]:
ratings.show()

### 1.2 Building ALS Model

In [ ]:
from pyspark.ml.recommendation import ALS

In [ ]:
train, test = ratings.randomSplit([0.9, 0.1], seed= 42)
print(train.count())
print(test.count())

In [ ]:
als = (ALS(nonnegative=True)
       .setUserCol("userID")
       .setItemCol("movieID")
       .setRatingCol("rating")
       .setColdStartStrategy("drop") # nan
      )
type(als)

**(nonnegative=True)**
By default, ALS does not enforce non-negativity constraints on the latent factors. This means that the factors (and hence the predicted ratings) can be negative. However, in many real-world scenarios, negative values may not make sense, especially if the ratings or preferences are inherently non-negative.

**.setColdStartStrategy()**

PySpark's ALS (Alternating Least Squares) model, the .setColdStartStrategy() method has two main options for handling cold-start problems:

>"drop": This is the most commonly used strategy. It drops the predictions for users or items that were not seen during training (i.e., cold-start users or items). This is useful because the model won’t make predictions for users or items it has no data for, avoiding unreliable predictions.

>"nan": This strategy returns NaN (Not a Number) for predictions of users or items that were not seen during training. Instead of removing these predictions, it marks them with NaN values so that you can handle them explicitly in post-processing. This might be useful if you need to track which predictions were unreliable or missing due to cold-start issues.

In [ ]:
print(als.explainParams())

In [ ]:
als_fitted = als.fit(train)

predict = als_fitted.transform(test)

predict.show(5)

In [ ]:
from pyspark.sql.functions import col
predict.filter(col('userID') == 0).show(5)

## 1.3 Evaluators

### 1.3.1 Regression Metrics

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
evaluator = (RegressionEvaluator()
             .setMetricName('rmse') # there is no trick that could be used to extract more than one metric, you can also use RegressionMetrics (below) directly
             .setPredictionCol('prediction')
             .setLabelCol('rating')
            )

# RMSE (Root Mean Squared Error) 

In [ ]:
evaluator.evaluate(predict) # 1.0018861045042096

### 1.2 Building ALS Model with tuning

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# create the parameter grid
params = (ParamGridBuilder()
          .addGrid(als.regParam, [.01, .08, .1,])
          .addGrid(als.rank, [10, 30]) # 
          .addGrid(als.maxIter, [10, 20])
          .build() # dont forget  .build()
)

If your machine or cluster has multiple CPU cores, you can set the setParallelism() value to a higher number. For example, if your cluster has 8 cores, you might set setParallelism(8).

.setNumFolds(b)
.setParallelism(a)
> If b = 3 (3 folds in cross-validation).

> If a = 5 (5 parallel tasks allowed).
In this case, since you only have 3 folds to evaluate, the extra 2 threads will be idle. The effective parallelism will be limited by the number of folds (b), not by a.

In [ ]:
# Instantiating cv estimator
cv = CrossValidator(
    estimator= als,
    estimatorParamMaps= params,
    evaluator = evaluator,
    parallelism= 3 # Evaluate up to 2 parameter settings in parallel
).setNumFolds(3)

# fitting
# cv_fitted = cv.fit(train)
# best_model = cv_fitted.bestModel
# type(best_model)

# predict_cv = best_model.transform(test)

# list(zip(best_model.avgMetrics, params))

In [ ]:
# fitting
cv_fitted = cv.fit(train)
best_model = cv_fitted.bestModel
type(best_model)

In [ ]:
predict_cv = best_model.transform(test)

In [ ]:
evaluator.evaluate(predict_cv) # 0.9744911114756902

In [ ]:
import numpy as np
cv_fitted.getEstimatorParamMaps()[np.argmin(cv_fitted.avgMetrics)]

# 0.1, 10, 20

## 1.3 Evaluators

### 1.3.2 RankingEvaluator

In [ ]:
from pyspark.ml.evaluation import RankingEvaluator

In [ ]:
# Generate top 10 recommendations for test data
predictions = predict

In [ ]:
predictions = predict_cv

In [ ]:
predictions.show(5)

In [ ]:
predictions.show(5)

In [ ]:
from pyspark.sql.functions import collect_list, col, desc
true_labels = predictions.filter(col('rating') > 1 ).orderBy("userId", desc("rating")).groupBy("userId").agg(collect_list("movieID").alias("true_items"))

# Predicted labels: items recommended by the model
predicted_labels = predictions.filter(col('prediction') > 1 ).orderBy("userId", desc("prediction")).groupBy("userId").agg(collect_list("movieID").alias("predicted_items"))

# Join true and predicted labels into a single DataFrame for evaluation
eval_df = (true_labels.join(predicted_labels, "userId")
           .withColumn("true_items", col("true_items").cast("array<double>"))
           .withColumn("predicted_items", col("predicted_items").cast("array<double>"))
          ) # Cast true_items and predicted_items to array<double>

In [ ]:
true_labels.show(5)

In [ ]:
predicted_labels.show(5)

In [ ]:
eval_df.show(5, False)

In [ ]:
# Initialize RankingEvaluator for Mean Average Precision at K (MAP@K)
evaluator = RankingEvaluator(
    predictionCol="predicted_items",
    labelCol="true_items",
    metricName="meanAveragePrecisionAtK",
    k= 3  # You can specify the value of K (e.g., top 10 recommendations)
)

In [ ]:
ndcg_at_k = evaluator.evaluate(eval_df)
print(f"NDCG at K: {ndcg_at_k}")

# NDCG at K = 4: 0.8898809523809524 for als_without_cv
# NDCG at K = 4: 0.9047619047619048 for als_with_cv

als_without_cv
> (k=3) 0.8888888888888888
> (k=5) 0.8994047619047618

als_with_cv
> (k=3) 0.9285714285714285
> (k=5) 0.9142857142857143 

In [ ]:
userRecs = als_fitted.recommendForAllUsers(10) 

# als_fitted
# best_model

In [ ]:
userRecs.sort(col('userID')).show(20, False)

In [ ]:
itemRecs = als_fitted.recommendForAllItems(10) 

In [ ]:
itemRecs.sort(col('movieID')).show(20, False)

In [ ]:
item_subset = [1, 3]  # Example subset of item IDs
item_subset_df = spark.createDataFrame([(movieID,) for movieID in item_subset], ["movieID"])

# Generate recommendations for the subset of items
num_users = 5  # Number of top users to recommend for each item
recommendations_Subset = als_fitted.recommendForItemSubset(item_subset_df, num_users)

In [ ]:
item_subset_df.show(5)

In [ ]:
recommendations_Subset.show(5, False)